# auto

> One `Chat` for both backends. Name a model, and rishi works out whether it needs litert or llama.cpp - the way [fastllm](https://github.com/AnswerDotAI/fastllm) picks a vendor from the model name.

In [ ]:
#| default_exp auto

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import asyncio
from importlib import import_module
from fastcore.all import Path, GetAttr

## Resolving a backend

rishi has two backends that take different model files: litert wants a `.litertlm` build, llama.cpp wants a `.gguf`. Both already accept the same `model_id`/`model_path` arguments, so the only thing standing between them and one front door is working out which is which.

Resolution follows [fastllm](https://github.com/AnswerDotAI/fastllm)'s order, most explicit first:

1. an explicit `backend='litert'|'llama'` argument;
2. a `backend/` prefix on the name, e.g. `'llama/unsloth/gemma-3-4b-it-GGUF'` - like fastllm's `split_vendor`, the prefix only counts if it names a known backend, so a real repo id such as `litert-community/gemma-4-E2B-it-litert-lm` is left alone;
3. the shape of the id or path - `.litertlm`, `litert-community` and `litert-lm` mean litert; `.gguf` and `GGUF` mean llama.cpp;
4. with nothing at all to go on, the default backend (litert, so a bare `Chat()` behaves as it always has).

Anything genuinely ambiguous raises rather than guessing, and says how to disambiguate. There is deliberately no table of friendly aliases: both backends already take full hub repo ids, and those resolve on their own.

In [ ]:
#| export
backends = {'litert': 'rishi.litert', 'llama': 'rishi.llama'}
dflt_backend = 'litert'

_pats = {'litert': ('.litertlm', 'litertlm', 'litert-community', 'litert-lm'), 'llama': ('.gguf', 'gguf')}

def split_backend(model):
    "Split `'backend/model'` into `(backend, model)`; the prefix must name a known backend, else `(None, model)`."
    if isinstance(model, str) and '/' in model:
        b, m = model.split('/', 1)
        if b in backends: return b, m
    return None, model

def infer_backend(model):
    "Guess a backend from the shape of a model id or path (`.litertlm` vs `.gguf`), else `None`."
    s = str(model or '').lower()
    if not s: return None
    return next((b for b, ps in _pats.items() if any(p in s for p in ps)), None)

def resolve_backend(model=None, backend=None, model_path=None):
    "Resolve `(backend, model)` from an explicit `backend`, a `backend/` prefix, or the id/path shape."
    pre, model = split_backend(model)
    nm = backend or pre or infer_backend(model) or infer_backend(model_path)
    if nm is None and model is None and model_path is None: nm = dflt_backend
    if nm is None: raise ValueError(
        f"Can't tell which backend {model!r} needs. Pass backend='litert'|'llama', prefix the name "
        f"(e.g. 'llama/{model}'), or give a full repo id or path (.litertlm / .gguf).")
    if nm not in backends: raise ValueError(f"Unknown backend {nm!r}; known backends: {', '.join(backends)}.")
    return nm, model

def get_backend(nm):
    "Import the backend module for `nm`, with an actionable error when its dependencies are missing."
    try: return import_module(backends[nm])
    except ImportError as e:
        extra = " Install it with: pip install 'rishi[llama]'" if nm == 'llama' else ''
        raise ImportError(f"The {nm!r} backend is unavailable ({e}).{extra}") from None

In [ ]:
from fastcore.test import test_eq, test_fail

In [ ]:
# a `backend/` prefix wins, but only when it really names a backend
test_eq(split_backend('llama/Qwen/Qwen3-4B-GGUF'), ('llama', 'Qwen/Qwen3-4B-GGUF'))
test_eq(split_backend('litert/gemma-4-E2B'), ('litert', 'gemma-4-E2B'))
test_eq(split_backend('litert-community/gemma-4-E2B-it-litert-lm'),
        (None, 'litert-community/gemma-4-E2B-it-litert-lm'))   # not a prefix - a real owner
test_eq(split_backend('plain-name'), (None, 'plain-name'))

# the shape of a full repo id or a path is enough on its own
test_eq(infer_backend('litert-community/gemma-4-E2B-it-litert-lm'), 'litert')
test_eq(infer_backend('Qwen/Qwen3-0.6B-GGUF'), 'llama')
test_eq(infer_backend('ggml-org/gemma-3-4b-it-GGUF'), 'llama')
test_eq(infer_backend('/models/model.litertlm'), 'litert')
test_eq(infer_backend('/models/model.gguf'), 'llama')
test_eq(infer_backend('gemma-4-E2B'), None)
test_eq(infer_backend(None), None)

In [ ]:
# resolution order: explicit argument, then prefix, then shape, then the default
test_eq(resolve_backend('Qwen/Qwen3-0.6B-GGUF'), ('llama', 'Qwen/Qwen3-0.6B-GGUF'))
test_eq(resolve_backend('litert-community/gemma-4-E2B-it-litert-lm'),
        ('litert', 'litert-community/gemma-4-E2B-it-litert-lm'))
test_eq(resolve_backend('llama/my-model'), ('llama', 'my-model'))          # prefix beats shape
test_eq(resolve_backend('Qwen/Qwen3-0.6B-GGUF', backend='litert')[0], 'litert')  # explicit beats all
test_eq(resolve_backend(model_path='/models/m.gguf'), ('llama', None))     # a path is enough
test_eq(resolve_backend(), ('litert', None))                              # bare Chat() is unchanged

# ambiguity raises, and says how to fix it
test_fail(lambda: resolve_backend('gemma-4-E2B'), contains="Can't tell which backend")
test_fail(lambda: resolve_backend('x', backend='mlx'), contains='Unknown backend')

# backends load lazily by name
import rishi.litert, rishi.llama
test_eq(get_backend('litert'), rishi.litert)
test_eq(get_backend('llama'), rishi.llama)

## Chat and AsyncChat

`Chat` is a factory, not a class: it resolves the backend and returns that backend's own `Chat`, so everything downstream - callbacks, tools, `hist`, `use`, streaming - is exactly the object documented in [core](core.html) or [llama](llama.html), with no wrapper in the way. Extra keyword arguments pass straight through, which also means backend-specific ones (`mmproj=`, `n_gpu_layers=`, `backend=Backend.GPU()`) still work.

`AsyncChat` gives both backends the same async surface by running blocking calls in a worker thread. Pass it a model, or an already-built `Chat` of either kind.

In [ ]:
#| export
def _is_path(model):
    "Does `model` name a local file rather than a hub repo id?"
    return bool(model) and (str(model).lower().endswith(('.gguf', '.litertlm')) or Path(model).exists())

def Chat(model=None, backend=None, **kw):
    "A `Chat` on whichever backend `model` implies; `**kw` passes through to that backend untouched."
    nm, model = resolve_backend(model, backend, kw.get('model_path'))
    if model is not None: kw.setdefault('model_path' if _is_path(model) else 'model_id', model)
    return get_backend(nm).Chat(**kw)

class AsyncChat(GetAttr):
    "Async twin of `Chat` for either backend; blocking calls run in a worker thread."
    _default = 'chat'
    def __init__(self, model=None, backend=None, **kw):
        self.chat = model if hasattr(model, '_send') else Chat(model, backend, **kw)
    async def __call__(self, msg=None, stream=False, max_output_tokens=None,
                       cbs=None # extra callbacks for this turn only
    ):
        'Run one chat turn; `await` the result (an async chunk generator when `stream=True`).'
        if stream: return self._astream(msg, max_output_tokens, cbs)
        return await asyncio.to_thread(self.chat, msg, False, max_output_tokens, cbs)
    async def _astream(self, msg, max_output_tokens=None, cbs=None):
        g = self.chat(msg, stream=True, max_output_tokens=max_output_tokens, cbs=cbs)
        done = object()
        while (o := await asyncio.to_thread(next, g, done)) is not done: yield o
    def close(self): self.chat.close()
    async def __aenter__(self): return self
    async def __aexit__(self, *exc): self.close()

In [ ]:
# the factory picks the class without loading any weights
import rishi.core, rishi.llama
from unittest.mock import patch as mock_patch

with mock_patch.object(rishi.llama.Chat, '__init__', return_value=None) as m:
    c = Chat('Qwen/Qwen3-0.6B-GGUF', think=False)
    assert isinstance(c, rishi.llama.Chat)
    test_eq(m.call_args.kwargs, {'model_id': 'Qwen/Qwen3-0.6B-GGUF', 'think': False})

with mock_patch.object(rishi.core.Chat, '__init__', return_value=None) as m:
    c = Chat('litert-community/gemma-4-E2B-it-litert-lm', sp='hi')
    assert isinstance(c, rishi.core.Chat)
    test_eq(m.call_args.kwargs, {'model_id': 'litert-community/gemma-4-E2B-it-litert-lm', 'sp': 'hi'})

# a local file goes to `model_path`, not `model_id`
with mock_patch.object(rishi.llama.Chat, '__init__', return_value=None) as m:
    Chat('/models/mine.gguf')
    test_eq(m.call_args.kwargs, {'model_path': '/models/mine.gguf'})

# a bare Chat() still means litert, and an explicit backend overrides the name
with mock_patch.object(rishi.core.Chat, '__init__', return_value=None) as m:
    Chat(); test_eq(m.call_args.kwargs, {})
with mock_patch.object(rishi.core.Chat, '__init__', return_value=None) as m:
    Chat('Qwen/Qwen3-0.6B-GGUF', backend='litert')
    test_eq(m.call_args.kwargs, {'model_id': 'Qwen/Qwen3-0.6B-GGUF'})

In [ ]:
# AsyncChat wraps an existing Chat of either backend, and proxies attributes to it
class _FakeChat:
    hist = ['x']
    def _send(self, *a, **k): pass
    def __call__(self, msg=None, stream=False, max_output_tokens=None, cbs=None):
        return iter(['a', 'b']) if stream else {'role': 'assistant', 'content': msg}
    def close(self): self.closed = True

from rishi.core import run_coro
ac = AsyncChat(_FakeChat())
test_eq(run_coro(ac('hi')), {'role': 'assistant', 'content': 'hi'})
test_eq(ac.hist, ['x'])                                   # GetAttr proxies to the wrapped chat

async def _collect():
    return [c async for c in await ac('go', stream=True)]
test_eq(run_coro(_collect()), ['a', 'b'])

## Using it

One import, either backend, same call.

In [ ]:
#| eval: false
from rishi.auto import Chat, AsyncChat
from rishi.core import resp_text

for m in ['litert-community/gemma-4-E2B-it-litert-lm',   # -> litert
          'Qwen/Qwen3-0.6B-GGUF']:                       # -> llama.cpp
    chat = Chat(m)
    print(f"{m:>45s} -> {resp_text(chat('Say hello in French.')).strip()[:40]}")
    chat.close()

In [ ]:
#| eval: false
# backend-specific arguments still pass straight through
gpu  = Chat('litert-community/gemma-4-E2B-it-litert-lm', backend=Backend.GPU(), cache_dir='.cache/litertlm')
vis  = Chat('ggml-org/gemma-3-4b-it-GGUF', mmproj=True, n_gpu_layers=-1)

# force a backend when the name alone can't say, or point at a local file
local = Chat('/models/my-finetune.gguf')
forced = Chat('my-org/private-build', backend='llama')

# and the async surface is the same on both
achat = AsyncChat('Qwen/Qwen3-0.6B-GGUF')
print(resp_text(await achat('Say hello.')))
async for c in await achat('Count to three.', stream=True): print(c, end='')

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()